In [ ]:
# ============================================================
# NB_TRNSF_GoldVentas_DimCliente
# Capa: Gold | Modelo: ventas | Tabla: d_cliente
# Fuente: Ih_silver.SFTP.Person
# Carga total — overwrite
# Todo desde config — sin hardcodeos
# ============================================================
from pyspark.sql import functions as F

# --- IDs técnicos (inmutables del TP) ---
# ============================================================
# IDs de workspace y lakehouses — SIN hardcodear (TP)
# ============================================================
# lakehouse.get() funciona en pipeline y en ejecución manual.
# El pipeline setea lh_gold como defaultLakehouse de los notebooks Gold.
_lh_gold   = notebookutils.lakehouse.get('lh_gold')
_lh_silver = notebookutils.lakehouse.get('lh_silver')
WS_ID      = _lh_gold.workspaceId
ART_GOLD   = _lh_gold.id
ART_SILVER = _lh_silver.id

# Leer config Silver — construye path origen dinámicamente
config_cli = spark.sql("""
    SELECT origen, nombre_tabla
    FROM lh_config.dbo.origin_bronze_silver
    WHERE origen = 'SFTP' AND nombre_tabla = 'Person' AND activo = 1
    LIMIT 1
""").collect()
if not config_cli:
    raise Exception("Config no encontrada para SFTP/Person en origin_bronze_silver")
_origen_cli = config_cli[0]['origen']
_tabla_cli  = config_cli[0]['nombre_tabla']

# FIX D01/D02: leer modelo y nombre_tabla Gold desde origin_gold
_og_cli = spark.sql("""
    SELECT modelo, nombre_tabla
    FROM lh_config.dbo.origin_gold
    WHERE nombre_notebook = 'NB_TRNSF_GoldVentas_DimCliente' AND activo = 1
    LIMIT 1
""").collect()
if not _og_cli:
    raise Exception("Config no encontrada para DimCliente en origin_gold")
_modelo_cli      = _og_cli[0]['modelo']       # 'ventas'
_tabla_gold_cli  = _og_cli[0]['nombre_tabla'] # 'd_cliente'

abfs_silver = (
    f"abfss://{WS_ID}@onelake.dfs.fabric.microsoft.com"
    f"/{ART_SILVER}/Tables/{_origen_cli}/{_tabla_cli}"
)
abfs_gold = (
    f"abfss://{WS_ID}@onelake.dfs.fabric.microsoft.com"
    f"/{ART_GOLD}/Tables/{_modelo_cli}/{_tabla_gold_cli}"
)

print(f"✅ Paths configurados desde config (origin_bronze_silver + origin_gold)")
print(f"   origen Silver : {_origen_cli} | tabla : {_tabla_cli}")
print(f"   modelo Gold   : {_modelo_cli} | tabla : {_tabla_gold_cli}")
print(f"   silver : {abfs_silver}")
print(f"   gold   : {abfs_gold}")


In [ ]:
# ============================================================
# Leer desde Silver
# ============================================================
# Leer via catálogo Spark — más robusto que path ABFS desde notebook Gold
df_silver = spark.sql(f"SELECT * FROM lh_silver.{_origen_cli}.{_tabla_cli} LIMIT 1000")
print(f"✅ Filas Silver Person: {df_silver.count()}")
display(df_silver.limit(5))

if df_silver.count() == 0:
    raise Exception("❌ Silver Person tiene 0 filas — ejecutar NB_TRNSF_SilverSFTP_Person primero")


In [ ]:
# ============================================================
# Transformar a dimensión d_cliente
# ============================================================
df_dim = (df_silver
    .withColumnRenamed("BusinessEntityID", "id_cliente")
    .withColumn("nombre_completo",
        F.concat_ws(" ",
            F.col("FirstName"),
            F.when(F.col("MiddleName").isNotNull(), F.col("MiddleName")).otherwise(F.lit("")),
            F.col("LastName")
        )
    )
    .withColumnRenamed("PersonType",      "tipo_persona")
    .withColumnRenamed("EmailPromotion",  "acepta_email_promo")
    .withColumnRenamed("ModifiedDate",    "fecha_modificacion")
    .select(
        "id_cliente", "nombre_completo",
        "tipo_persona", "acepta_email_promo",
        "fecha_modificacion"
    )
)

print(f"✅ Filas dim_cliente: {df_dim.count()}")
display(df_dim.select("id_cliente","nombre_completo","tipo_persona","acepta_email_promo").limit(10))


In [ ]:
# ============================================================
# Guardar en Gold — carga total
# ============================================================
try:
    notebookutils.fs.rm(abfs_gold, recurse=True)
    print(f"🗑️  Tabla anterior eliminada")
except:
    print("ℹ️  Primera carga")

(df_dim.write
    .format("delta")
    .mode("overwrite")
    .option("overwriteSchema", "true")
    .save(abfs_gold))

print(f"✅ d_cliente guardada en Gold: {df_dim.count()} filas")


In [ ]:
# ============================================================
# Verificar resultado
# ============================================================
df_ver = spark.read.format("delta").load(abfs_gold)
print(f"✅ Total filas en Gold d_cliente: {df_ver.count()}")
display(df_ver
    .select("id_cliente","nombre_completo","tipo_persona","acepta_email_promo")
    .orderBy("id_cliente")
    .limit(10))
